In [2]:
# ── CELL 1: Imports and Path Configuration ─────────────────────────────────────
# Notebook 05a — Local MD Preparation (GROMACS input generation)
# PfDHFR-TS K1 (PDB: 1J3I) | 3 hits + pyrimethamine reference
# Kenneth Odoh Chidiebere — MSc Computational Drug Discovery
#
# Environment : cheminfo (Python 3.11)
# External tools called via subprocess:
#   - obabel.exe    (openbabel_env) — PDBQT → PDB conversion
#   - acpype        (separate install — see preflight notes below)
#
# Compounds for MD:
#   CNP0286261.0  African NP  -11.86 kcal/mol  (dock_id: CNP0286261_0)
#   CNP0275186.1  African NP  -11.13 kcal/mol  (dock_id: CNP0275186_1)
#   CNP0539885.2  Global NP   -10.03 kcal/mol  (dock_id: CNP0539885_2)
#   Pyrimethamine Reference   - 6.90 kcal/mol  (dock_id: pyrimethamine_ref)

import os
import sys
import subprocess
import shutil
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolTransforms

# ── Project paths ──────────────────────────────────────────────────────────────

BASE    = Path(r"C:\my_projects_all\portfolio_projects\Msc_project")
DATA    = BASE / "data"
FIGURES = BASE / "figures"

DOCK_DIR    = DATA / "docking"
RESULTS_DIR = DOCK_DIR / "results"

MD_DIR      = DATA / "md"
POSES_DIR   = MD_DIR / "poses"          # extracted best-mode PDB files
LIGAND_DIR  = MD_DIR / "ligand_params"  # ACPYPE output (per compound)
SYSTEM_DIR  = MD_DIR / "systems"        # assembled GROMACS system (per compound)
MDP_DIR     = MD_DIR / "mdp_files"     # GROMACS run parameter files
COLAB_DIR   = MD_DIR / "colab_upload"  # zipped packages for Google Drive

for d in [MD_DIR, POSES_DIR, LIGAND_DIR, SYSTEM_DIR, MDP_DIR, COLAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── External tool paths ────────────────────────────────────────────────────────

OBABEL_EXE = Path(
    r"C:\Users\Kenjo\anaconda3\envs\openbabel_env\Library\bin\obabel.exe"
)

# ── MD compound manifest ───────────────────────────────────────────────────────

MD_COMPOUNDS = [
    {
        "name"       : "CNP0286261_0",
        "identifier" : "CNP0286261.0",
        "smiles"     : "O=C(CCc1ccccc1)c1c(O)c(Cc2ccccc2O)c2c(c1O)Cc1ccccc1O2",
        "tier"       : "African NP",
        "vina_score" : -11.86,
        "pdbqt_out"  : RESULTS_DIR / "CNP0286261_0_out.pdbqt",
    },
    {
        "name"       : "CNP0275186_1",
        "identifier" : "CNP0275186.1",
        "smiles"     : "CC(C)C1=CC2=CC=C3[C@@](C)(COC(=O)c4ccc(O)c(O)c4)CCC[C@]3(C)C2=C(O)C1=O",
        "tier"       : "African NP",
        "vina_score" : -11.13,
        "pdbqt_out"  : RESULTS_DIR / "CNP0275186_1_out.pdbqt",
    },
    {
        "name"       : "CNP0539885_2",
        "identifier" : "CNP0539885.2",
        "smiles"     : "N=c1nc(O)c2c(CCc3ccc(C(=O)N[C@@H](CCC(=O)O)C(=O)O)cc3)c[nH]c2[nH]1",
        "tier"       : "Global NP",
        "vina_score" : -10.03,
        "pdbqt_out"  : RESULTS_DIR / "CNP0539885_2_out.pdbqt",
    },
    {
        "name"       : "pyrimethamine",
        "identifier" : "pyrimethamine",
        "smiles"     : "Cc1cnc(N)nc1-c1ccc(Cl)cc1",
        "tier"       : "Reference",
        "vina_score" : -6.90,
        "pdbqt_out"  : RESULTS_DIR / "ref_pyrimethamine_out.pdbqt",
    },
]

# ── Logging ────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler(MD_DIR / "md_prep.log"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger(__name__)

# ── Preflight checks ───────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("CELL 1 — IMPORTS AND CONFIGURATION")
print("=" * 60)

# OpenBabel
status = "✓" if OBABEL_EXE.exists() else "✗  NOT FOUND"
print(f"\n  {status}  OpenBabel  {OBABEL_EXE}")

# ACPYPE — must be importable in cheminfo env
try:
    result = subprocess.run(
        ["acpype", "--version"],
        capture_output=True, text=True, timeout=10
    )
    ver = result.stdout.strip() or result.stderr.strip()
    print(f"  ✓  acpype  — {ver}")
except FileNotFoundError:
    print("  ✗  acpype  — NOT FOUND in PATH")
    print("     Install with:  pip install acpype  (in cheminfo env)")
    print("     Then restart kernel and re-run this cell.")
except Exception as e:
    print(f"  ✗  acpype  — {e}")

# Docked output files
print(f"\n  Checking docked output files:")
all_present = True
for cpd in MD_COMPOUNDS:
    exists = cpd["pdbqt_out"].exists()
    flag   = "✓" if exists else "✗  MISSING"
    print(f"    {flag}  {cpd['pdbqt_out'].name}")
    if not exists:
        all_present = False

if all_present:
    print(f"\n  All docked outputs present — ready to proceed.")
else:
    print(f"\n  ✗  One or more docked output files missing.")
    print(f"     Confirm file names in:  {RESULTS_DIR}")

# Directory summary
print(f"\n  MD working directory  : {MD_DIR}")
print(f"  Poses output          : {POSES_DIR}")
print(f"  Ligand params         : {LIGAND_DIR}")
print(f"  GROMACS systems       : {SYSTEM_DIR}")
print(f"  MDP files             : {MDP_DIR}")
print(f"  Colab upload package  : {COLAB_DIR}")

print(f"\n  Compounds for MD:")
for cpd in MD_COMPOUNDS:
    print(f"    {cpd['name']:<20}  {cpd['tier']:<12}  "
          f"{cpd['vina_score']:>7.2f} kcal/mol")

print(f"\n{'='*60}")
print(f"  Next cell: extract best docking poses from _out.pdbqt files")
print(f"{'='*60}")


CELL 1 — IMPORTS AND CONFIGURATION

  ✓  OpenBabel  C:\Users\Kenjo\anaconda3\envs\openbabel_env\Library\bin\obabel.exe
  ✓  acpype  — ============================================================================
| ACPYPE: AnteChamber PYthon Parser interfacE v. 2023.10.27 (c) 2026 AWSdS |

  Checking docked output files:
    ✓  CNP0286261_0_out.pdbqt
    ✓  CNP0275186_1_out.pdbqt
    ✓  CNP0539885_2_out.pdbqt
    ✓  ref_pyrimethamine_out.pdbqt

  All docked outputs present — ready to proceed.

  MD working directory  : C:\my_projects_all\portfolio_projects\Msc_project\data\md
  Poses output          : C:\my_projects_all\portfolio_projects\Msc_project\data\md\poses
  Ligand params         : C:\my_projects_all\portfolio_projects\Msc_project\data\md\ligand_params
  GROMACS systems       : C:\my_projects_all\portfolio_projects\Msc_project\data\md\systems
  MDP files             : C:\my_projects_all\portfolio_projects\Msc_project\data\md\mdp_files
  Colab upload package  : C:\my_projects_all

In [3]:
# ── CELL 2: Extract Best Docking Poses from PDBQT Output Files ─────────────────
# Strategy: extract heavy-atom coordinates from Vina _out.pdbqt (MODE 1),
# then use RDKit to assign bond topology from SMILES and embed hydrogens
# constrained to the docked heavy-atom positions.
# This is the standard approach for GROMACS/ACPYPE input preparation.
#
# Key fixes documented here:
#   - Element parsed from atom-name field (cols 12-15), NOT cols 77-78
#     which contain AutoDock atom types (A, OA, NA …) not element symbols
#   - Polar H atoms in PDBQT excluded before sequential assignment
#   - H positions placed by RDKit MMFF minimisation constrained to heavy atoms

print("\n" + "=" * 60)
print("CELL 2 — EXTRACT BEST DOCKING POSES")
print("=" * 60)

from rdkit.Chem import AllChem


def extract_mode1_coords(pdbqt_path):
    """
    Parse MODE 1 from a Vina _out.pdbqt file.
    Element is read from the atom-name field (cols 12-15), NOT cols 76-78
    which contain AutoDock atom types (A, OA, NA, HD …) not element symbols.
    Hydrogen lines (element == H) are excluded so that sequential assignment
    onto RDKit heavy atoms is correctly aligned.
    Returns list of (serial, element, x, y, z) for heavy atoms only.
    """
    lines  = Path(pdbqt_path).read_text(encoding='utf-8').splitlines()
    coords = []
    in_m1  = False

    for line in lines:
        tag = line[:6].strip()
        if tag == "MODEL":
            if int(line.split()[1]) == 1:
                in_m1 = True
            else:
                break
        if not in_m1:
            continue
        if tag in ("ATOM", "HETATM"):
            try:
                serial    = int(line[6:11])
                atom_name = line[12:16].strip().lstrip("0123456789")
                element   = (atom_name[0].upper() + atom_name[1:].lower()
                             if len(atom_name) > 1 else atom_name.upper())
                if element == "H":
                    continue        # skip polar Hs added by Meeko
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append((serial, element, x, y, z))
            except (ValueError, IndexError):
                continue
        if tag == "ENDMDL" and in_m1:
            break
    return coords


def embed_hs_on_docked_pose(smiles, docked_coords, compound_name):
    """
    Build a fully protonated RDKit mol with docked heavy-atom positions.

    Steps:
    1. Parse SMILES → assign bond topology
    2. Add explicit Hs
    3. Generate a fresh 3D conformer (ETKDGv3) — needed for H placement
    4. Overwrite heavy-atom positions with docked coordinates
    5. Optimise only H positions (constrain all heavy atoms)
    Returns RDKit mol with conformer, or None on failure.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        logger.error(f"  {compound_name}: invalid SMILES")
        return None

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    result = AllChem.EmbedMolecule(mol, params)
    if result == -1:
        logger.error(f"  {compound_name}: ETKDGv3 embedding failed")
        return None

    conf          = mol.GetConformer()
    heavy_indices = [i for i, a in enumerate(mol.GetAtoms())
                     if a.GetAtomicNum() != 1]

    if len(heavy_indices) != len(docked_coords):
        logger.warning(
            f"  {compound_name}: heavy atom count mismatch — "
            f"RDKit={len(heavy_indices)}, PDBQT={len(docked_coords)}. "
            f"Using sequential assignment."
        )
    n_assign = min(len(heavy_indices), len(docked_coords))
    for i in range(n_assign):
        _, elem, x, y, z = docked_coords[i]
        conf.SetAtomPosition(heavy_indices[i], (x, y, z))

    # Minimise H positions only — constrain all heavy atoms in place
    ff = AllChem.MMFFGetMoleculeForceField(
        mol,
        AllChem.MMFFGetMoleculeProperties(mol),
        confId=0
    )
    if ff is None:
        ff = AllChem.UFFGetMoleculeForceField(mol, confId=0)

    if ff is not None:
        for idx in heavy_indices:
            ff.AddFixedPoint(idx)
        ff.Minimize(maxIts=500)

    return mol


def write_mol_to_pdb(mol, out_path, compound_name):
    """Write RDKit mol with conformer to PDB file."""
    from rdkit.Chem import PDBWriter
    writer = PDBWriter(str(out_path))
    writer.write(mol)
    writer.close()


# ── Process all compounds ──────────────────────────────────────────────────────

print(f"\n  Extracting MODE 1 heavy-atom coords, embedding Hs via RDKit ...\n")

pose_summary = []

for cpd in MD_COMPOUNDS:
    name     = cpd["name"]
    pdbqt_in = cpd["pdbqt_out"]
    pdb_out  = POSES_DIR / f"{name}_pose.pdb"

    docked_coords = extract_mode1_coords(pdbqt_in)
    if not docked_coords:
        print(f"  ✗  {name}  — no coordinates extracted from PDBQT")
        cpd["pdb_pose"] = None
        pose_summary.append({"name": name, "tier": cpd["tier"],
                              "vina_score": cpd["vina_score"],
                              "pdb_path": "FAILED", "success": False})
        continue

    mol = embed_hs_on_docked_pose(cpd["smiles"], docked_coords, name)
    if mol is None:
        print(f"  ✗  {name}  — RDKit embedding failed")
        cpd["pdb_pose"] = None
        pose_summary.append({"name": name, "tier": cpd["tier"],
                              "vina_score": cpd["vina_score"],
                              "pdb_path": "FAILED", "success": False})
        continue

    write_mol_to_pdb(mol, pdb_out, name)

    n_atoms = mol.GetNumAtoms()
    n_heavy = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() != 1)
    n_h     = n_atoms - n_heavy
    size_kb = pdb_out.stat().st_size / 1024

    print(f"  ✓  {name:<22}  {n_heavy} heavy + {n_h} H = {n_atoms} atoms"
          f"  ({size_kb:.1f} KB)  → {pdb_out.name}")

    cpd["pdb_pose"] = pdb_out
    pose_summary.append({"name": name, "tier": cpd["tier"],
                          "vina_score": cpd["vina_score"],
                          "pdb_path": str(pdb_out), "success": True})

# ── Summary ────────────────────────────────────────────────────────────────────

n_ok = sum(r["success"] for r in pose_summary)
print(f"\n  Poses prepared  : {n_ok} / {len(pose_summary)}")

pose_df = pd.DataFrame(pose_summary)
pose_df.to_csv(MD_DIR / "pose_manifest.csv", index=False)
print(f"  Manifest saved  : {MD_DIR / 'pose_manifest.csv'}")

print(f"\n{'='*60}")
print(f"  Next cell: validate PDB files before ACPYPE")
print(f"{'='*60}")


CELL 2 — EXTRACT BEST DOCKING POSES

  Extracting MODE 1 heavy-atom coords, embedding Hs via RDKit ...

  ✓  CNP0286261_0            34 heavy + 24 H = 58 atoms  (5.4 KB)  → CNP0286261_0_pose.pdb
  ✓  CNP0275186_1            33 heavy + 30 H = 63 atoms  (5.8 KB)  → CNP0275186_1_pose.pdb
  ✓  CNP0539885_2            31 heavy + 21 H = 52 atoms  (4.8 KB)  → CNP0539885_2_pose.pdb
  ✓  pyrimethamine           15 heavy + 10 H = 25 atoms  (2.3 KB)  → pyrimethamine_pose.pdb

  Poses prepared  : 4 / 4
  Manifest saved  : C:\my_projects_all\portfolio_projects\Msc_project\data\md\pose_manifest.csv

  Next cell: validate PDB files before ACPYPE


In [4]:
# ── CELL 3: Inspect and Validate Pose PDB Files ────────────────────────────────

print("\n" + "=" * 60)
print("CELL 3 — VALIDATE POSE PDB FILES")
print("=" * 60)


def count_pdb_atoms(pdb_path):
    """Count ATOM/HETATM lines, element types, and check for END record."""
    lines    = Path(pdb_path).read_text(encoding='utf-8').splitlines()
    atoms    = [l for l in lines if l.startswith(("ATOM", "HETATM"))]
    has_end  = any(l.strip() == "END" for l in lines)
    elements = {}
    for l in atoms:
        elem = l[76:78].strip() if len(l) > 76 else l[12:14].strip().lstrip("0123456789")
        elem = elem.capitalize()
        elements[elem] = elements.get(elem, 0) + 1
    return len(atoms), elements, has_end


def rdkit_atom_count(smiles):
    """Return (n_heavy, n_total_with_H) from RDKit."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    mol_h = Chem.AddHs(mol)
    return mol.GetNumAtoms(), mol_h.GetNumAtoms()


# ── Validate each compound ─────────────────────────────────────────────────────

print()
all_valid = True

for cpd in MD_COMPOUNDS:
    name     = cpd["name"]
    pdb_path = cpd.get("pdb_pose")

    if pdb_path is None or not pdb_path.exists():
        print(f"  ✗  {name}  — PDB file missing, skipping")
        all_valid = False
        continue

    n_pdb, elements, has_end    = count_pdb_atoms(pdb_path)
    n_heavy_rdkit, n_total_rdkit = rdkit_atom_count(cpd["smiles"])

    delta = abs(n_pdb - n_total_rdkit) if n_total_rdkit else 999
    ok    = delta <= 3 and has_end

    status = "✓" if ok else "✗"
    print(f"  {status}  {name}")
    print(f"       PDB atoms      : {n_pdb}  (RDKit expects ~{n_total_rdkit}, "
          f"heavy={n_heavy_rdkit}, delta={delta})")

    elem_str = "  ".join(f"{e}:{c}" for e, c in sorted(elements.items()))
    print(f"       Elements       : {elem_str}")
    print(f"       END record     : {'yes' if has_end else 'NO — missing END'}")

    bad_elems = [e for e in elements if e not in
                 {"C","H","N","O","S","P","F","Cl","Br","I","Se","Si"}]
    if bad_elems:
        print(f"       ⚠  Unexpected elements: {bad_elems}")
        all_valid = False

    if not ok:
        all_valid = False

    print()

# ── Raw PDB excerpt ────────────────────────────────────────────────────────────

print("  ── Raw PDB excerpt (CNP0286261_0, first 8 ATOM lines) ─────────────")
pdb_lines  = Path(POSES_DIR / "CNP0286261_0_pose.pdb").read_text().splitlines()
atom_lines = [l for l in pdb_lines if l.startswith(("ATOM", "HETATM"))]
for l in atom_lines[:8]:
    print(f"    {l}")

# ── Summary ────────────────────────────────────────────────────────────────────

print(f"\n  {'All poses valid — ready for ACPYPE parametrisation.' if all_valid else 'Issues found — review above before proceeding.'}")

print(f"\n{'='*60}")
print(f"  Next cell: ACPYPE ligand parametrisation (GAFF2 + AM1-BCC)")
print(f"{'='*60}")


CELL 3 — VALIDATE POSE PDB FILES

  ✓  CNP0286261_0
       PDB atoms      : 58  (RDKit expects ~58, heavy=34, delta=0)
       Elements       : C:29  H:24  O:5
       END record     : yes

  ✓  CNP0275186_1
       PDB atoms      : 63  (RDKit expects ~63, heavy=33, delta=0)
       Elements       : C:27  H:30  O:6
       END record     : yes

  ✓  CNP0539885_2
       PDB atoms      : 52  (RDKit expects ~52, heavy=31, delta=0)
       Elements       : C:20  H:21  N:5  O:6
       END record     : yes

  ✓  pyrimethamine
       PDB atoms      : 25  (RDKit expects ~25, heavy=15, delta=0)
       Elements       : C:11  Cl:1  H:10  N:3
       END record     : yes

  ── Raw PDB excerpt (CNP0286261_0, first 8 ATOM lines) ─────────────
    HETATM    1  O1  UNL     1      30.631   6.878  63.110  1.00  0.00           O  
    HETATM    2  C1  UNL     1      30.732   5.748  62.282  1.00  0.00           C  
    HETATM    3  C2  UNL     1      29.603   5.238  61.604  1.00  0.00           C  
    HETATM  

In [5]:
# ── CELL 4: Validate ACPYPE Outputs and Extract Key Parameters ─────────────────
# ACPYPE parametrisation was performed on Google Colab (Linux) due to
# AmberTools not being available for Windows via conda-forge.
# This cell validates the downloaded ITP/GRO files and extracts atom type
# and charge information for methods documentation.

print("\n" + "=" * 60)
print("CELL 4 — VALIDATE ACPYPE OUTPUTS (GAFF2 + AM1-BCC)")
print("=" * 60)

def parse_itp(itp_path):
    """
    Extract key parameters from a GROMACS ITP file:
    - Number of atoms
    - Total charge (sum of partial charges)
    - Unique GAFF2 atom types
    - Number of bonds, angles, dihedrals
    """
    text     = itp_path.read_text(encoding='utf-8')
    sections = {}
    current  = None
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith(';'):
            continue
        if line.startswith('['):
            current = line.strip('[] ').strip()
            sections[current] = []
        elif current:
            sections[current].append(line)

    # Atoms section: idx, type, resnr, res, name, cgnr, charge, mass
    atoms       = sections.get('atoms', [])
    n_atoms     = len(atoms)
    total_charge = 0.0
    atom_types  = set()
    for a in atoms:
        parts = a.split()
        if len(parts) >= 8:
            try:
                total_charge += float(parts[6])
                atom_types.add(parts[1])
            except ValueError:
                pass

    n_bonds     = len(sections.get('bonds', []))
    n_angles    = len(sections.get('angles', []))
    n_dihedrals = len(sections.get('dihedrals', []))

    return {
        'n_atoms'      : n_atoms,
        'total_charge' : round(total_charge, 4),
        'atom_types'   : sorted(atom_types),
        'n_atom_types' : len(atom_types),
        'n_bonds'      : n_bonds,
        'n_angles'     : n_angles,
        'n_dihedrals'  : n_dihedrals,
    }


def parse_gro(gro_path):
    """Extract atom count and box vectors from GRO file."""
    lines = gro_path.read_text(encoding='utf-8').splitlines()
    n_atoms  = int(lines[1].strip()) if len(lines) > 1 else 0
    box_line = lines[-1].strip() if lines else ""
    return {'n_atoms': n_atoms, 'box_line': box_line}


# ── Validate all compounds ─────────────────────────────────────────────────────

print()
param_records = []
all_valid = True

for cpd in MD_COMPOUNDS:
    name    = cpd["name"]
    itp_path = LIGAND_DIR / name / f"{name}_GMX.itp"
    gro_path = LIGAND_DIR / name / f"{name}_GMX.gro"

    if not itp_path.exists() or not gro_path.exists():
        print(f"  ✗  {name}  — files missing")
        all_valid = False
        continue

    itp_info = parse_itp(itp_path)
    gro_info = parse_gro(gro_path)

    # Store paths back into manifest
    cpd["itp_path"] = itp_path
    cpd["gro_path"] = gro_path

    # Charge check — should be within 0.02 of formal charge
    formal_charge = sum(a.GetFormalCharge() for a in
                        Chem.MolFromSmiles(cpd["smiles"]).GetAtoms())
    charge_ok = abs(itp_info['total_charge'] - formal_charge) < 0.05
    atom_ok   = itp_info['n_atoms'] == gro_info['n_atoms']
    ok        = charge_ok and atom_ok

    status = "✓" if ok else "✗"
    print(f"  {status}  {name}  ({cpd['tier']})")
    print(f"       Atoms          : ITP={itp_info['n_atoms']}  "
          f"GRO={gro_info['n_atoms']}  "
          f"{'✓ match' if atom_ok else '✗ MISMATCH'}")
    print(f"       Total charge   : {itp_info['total_charge']:+.4f}  "
          f"(formal={formal_charge:+d})  "
          f"{'✓' if charge_ok else '✗ EXCEEDS TOLERANCE'}")
    print(f"       GAFF2 types    : {itp_info['n_atom_types']} unique  "
          f"→ {', '.join(itp_info['atom_types'][:8])}"
          f"{'...' if itp_info['n_atom_types'] > 8 else ''}")
    print(f"       Bonds/Angles/Dihedrals : "
          f"{itp_info['n_bonds']} / {itp_info['n_angles']} / "
          f"{itp_info['n_dihedrals']}")
    print()

    if not ok:
        all_valid = False

    param_records.append({
        "name"          : name,
        "tier"          : cpd["tier"],
        "n_atoms"       : itp_info["n_atoms"],
        "total_charge"  : itp_info["total_charge"],
        "formal_charge" : formal_charge,
        "n_gaff2_types" : itp_info["n_atom_types"],
        "n_bonds"       : itp_info["n_bonds"],
        "n_angles"      : itp_info["n_angles"],
        "n_dihedrals"   : itp_info["n_dihedrals"],
        "itp_path"      : str(itp_path),
        "gro_path"      : str(gro_path),
    })

# ── Save summary ───────────────────────────────────────────────────────────────

param_df = pd.DataFrame(param_records)
param_df.to_csv(MD_DIR / "acpype_summary.csv", index=False)
print(f"  Summary saved : {MD_DIR / 'acpype_summary.csv'}")
print(f"\n  {'All parameters valid — ready for system assembly.' if all_valid else 'Issues found — review above.'}")

print(f"\n{'='*60}")
print(f"  Next cell: receptor preparation (1J3I chain A → GROMACS PDB)")
print(f"{'='*60}")


CELL 4 — VALIDATE ACPYPE OUTPUTS (GAFF2 + AM1-BCC)

  ✓  CNP0286261_0  (African NP)
       Atoms          : ITP=58  GRO=58  ✓ match
       Total charge   : +0.0000  (formal=+0)  ✓
       GAFF2 types    : 10 unique  → c, c3, c6, ca, ha, hc, ho, o...
       Bonds/Angles/Dihedrals : 62 / 103 / 0

  ✓  CNP0275186_1  (African NP)
       Atoms          : ITP=63  GRO=63  ✓ match
       Total charge   : +0.0000  (formal=+0)  ✓
       GAFF2 types    : 15 unique  → c, c2, c3, c6, ca, cc, cd, ce...
       Bonds/Angles/Dihedrals : 66 / 118 / 0

  ✓  CNP0539885_2  (Global NP)
       Atoms          : ITP=52  GRO=52  ✓ match
       Total charge   : -0.0000  (formal=+0)  ✓
       GAFF2 types    : 17 unique  → c, c3, ca, cc, cd, h1, h4, ha...
       Bonds/Angles/Dihedrals : 54 / 89 / 0

  ✓  pyrimethamine  (Reference)
       Atoms          : ITP=25  GRO=25  ✓ match
       Total charge   : +0.0000  (formal=+0)  ✓
       GAFF2 types    : 10 unique  → c3, ca, cl, cp, h4, ha, hc, hn...
       Bonds/Angles

In [6]:
# ── CELL 5: Receptor Preparation for GROMACS ───────────────────────────────────
# Input : data/docking/receptor/1J3I_clean.pdb  (chain A, no HOH/NDP/WRA)
# Output: data/md/receptor/1J3I_chainA_GMX.pdb  (GROMACS-ready)
#
# GROMACS requires:
#   - Standard amino acid residue names (no AMBER/CHARMM variants)
#   - No HETATM records except for the ligand (handled separately)
#   - Consecutive atom and residue numbering
#   - TER record between chain end and END

print("\n" + "=" * 60)
print("CELL 5 — RECEPTOR PREPARATION")
print("=" * 60)

RECEPTOR_DIR_MD = MD_DIR / "receptor"
RECEPTOR_DIR_MD.mkdir(exist_ok=True)

# Source: cleaned PDB from Notebook 04
source_pdb = DATA / "docking" / "receptor" / "1J3I_clean.pdb"
output_pdb = RECEPTOR_DIR_MD / "1J3I_chainA_GMX.pdb"

print(f"\n  Source PDB : {source_pdb}")
print(f"  Exists     : {source_pdb.exists()}")

if not source_pdb.exists():
    print(f"\n  ✗  Source PDB not found.")
    print(f"     Expected at: {source_pdb}")
    print(f"     Check docking/receptor/ directory contents:")
    for f in (DATA / "docking" / "receptor").iterdir():
        print(f"       {f.name}")
else:
    # ── Parse and validate the clean PDB ──────────────────────────────────────
    lines = source_pdb.read_text(encoding='utf-8').splitlines()

    atom_lines   = [l for l in lines if l.startswith("ATOM")]
    hetatm_lines = [l for l in lines if l.startswith("HETATM")]
    ter_lines    = [l for l in lines if l.startswith("TER")]

    # Extract unique residue names and chains
    residues = set()
    chains   = set()
    for l in atom_lines:
        residues.add(l[17:20].strip())
        chains.add(l[21].strip())

    hetatm_resnames = set()
    for l in hetatm_lines:
        hetatm_resnames.add(l[17:20].strip())

    print(f"\n  ── Source PDB contents ─────────────────────────────────")
    print(f"  ATOM lines    : {len(atom_lines)}")
    print(f"  HETATM lines  : {len(hetatm_lines)}  "
          f"(residues: {hetatm_resnames if hetatm_resnames else 'none'})")
    print(f"  TER records   : {len(ter_lines)}")
    print(f"  Chains        : {sorted(chains)}")
    print(f"  Unique resnames: {len(residues)}  "
          f"({', '.join(sorted(residues)[:10])}"
          f"{'...' if len(residues) > 10 else ''})")

    # ── Write GROMACS-ready PDB ────────────────────────────────────────────────
    # Keep only ATOM lines (pure protein — no HETATM)
    # GROMACS pdb2gmx will add missing hydrogens and assign force field types
    gromacs_lines = []
    for l in lines:
        if l.startswith("ATOM"):
            gromacs_lines.append(l)
        elif l.startswith("TER"):
            gromacs_lines.append(l)
    gromacs_lines.append("END")

    output_pdb.write_text("\n".join(gromacs_lines) + "\n", encoding='utf-8')

    # ── Verify output ──────────────────────────────────────────────────────────
    out_lines = output_pdb.read_text().splitlines()
    n_atom_out = sum(1 for l in out_lines if l.startswith("ATOM"))
    has_end    = any(l.strip() == "END" for l in out_lines)
    has_ter    = any(l.startswith("TER") for l in out_lines)

    print(f"\n  ── GROMACS receptor PDB ────────────────────────────────")
    print(f"  ATOM lines    : {n_atom_out}")
    print(f"  TER record    : {'yes' if has_ter else 'NO — missing'}")
    print(f"  END record    : {'yes' if has_end else 'NO — missing'}")
    print(f"  File size     : {output_pdb.stat().st_size / 1024:.1f} KB")
    print(f"  Output path   : {output_pdb}")

    # ── Residue range check ────────────────────────────────────────────────────
    res_nums = []
    for l in out_lines:
        if l.startswith("ATOM"):
            try:
                res_nums.append(int(l[22:26]))
            except ValueError:
                pass
    if res_nums:
        print(f"  Residue range : {min(res_nums)} – {max(res_nums)}")

print(f"\n{'='*60}")
print(f"  Next cell: generate GROMACS topology with pdb2gmx (Colab)")
print(f"{'='*60}")


CELL 5 — RECEPTOR PREPARATION

  Source PDB : C:\my_projects_all\portfolio_projects\Msc_project\data\docking\receptor\1J3I_clean.pdb
  Exists     : True

  ── Source PDB contents ─────────────────────────────────
  ATOM lines    : 1847
  HETATM lines  : 0  (residues: none)
  TER records   : 1
  Chains        : ['A']
  Unique resnames: 19  (ALA, ARG, ASN, ASP, CYS, GLN, GLU, GLY, ILE, LEU...)

  ── GROMACS receptor PDB ────────────────────────────────
  ATOM lines    : 1847
  TER record    : yes
  END record    : yes
  File size     : 147.9 KB
  Output path   : C:\my_projects_all\portfolio_projects\Msc_project\data\md\receptor\1J3I_chainA_GMX.pdb
  Residue range : 1 – 233

  Next cell: generate GROMACS topology with pdb2gmx (Colab)


In [7]:
# ── CELL 6: Upload Receptor PDB to Google Drive for Colab pdb2gmx ──────────────
# GROMACS is Linux-only — pdb2gmx will be run on Google Colab.
# This cell copies the receptor PDB to the Drive folder structure
# so the Colab notebook can access it directly.

print("\n" + "=" * 60)
print("CELL 6 — STAGE RECEPTOR FOR COLAB pdb2gmx")
print("=" * 60)

import shutil

# ── Expected Drive folder structure ───────────────────────────────────────────
# My Drive/PfDHFR_MD/
#   poses/                  ← already populated (Cell 2)
#   acpype_output/          ← already populated (Colab Cell 2)
#   receptor/               ← populated here
#   pdb2gmx_output/         ← will be created by Colab

DRIVE_RECEPTOR_DIR = (
    Path(r"C:\Users\Kenjo\AppData\Local\Google\Drive")  # fallback if Drive for desktop
)

# Primary: copy to a staging folder inside the project for manual upload
STAGE_DIR = MD_DIR / "colab_upload" / "receptor"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

receptor_src = RECEPTOR_DIR_MD / "1J3I_chainA_GMX.pdb"
receptor_dst = STAGE_DIR / "1J3I_chainA_GMX.pdb"

shutil.copy(receptor_src, receptor_dst)

print(f"\n  Receptor PDB staged for Drive upload:")
print(f"  Source : {receptor_src}")
print(f"  Staged : {receptor_dst}")
print(f"  Size   : {receptor_dst.stat().st_size / 1024:.1f} KB")

print(f"""
  ── Manual upload instructions ──────────────────────────
  Upload this file to Google Drive at:
    My Drive/PfDHFR_MD/receptor/1J3I_chainA_GMX.pdb

  The Colab pdb2gmx notebook will read from this path.
  ────────────────────────────────────────────────────────
""")

# ── Document expected Colab outputs ───────────────────────────────────────────
print("  Expected outputs from Colab pdb2gmx run:")
print("  (to be downloaded into data/md/receptor/ locally)")
print()
expected = [
    ("1J3I_processed.gro",  "Protein coordinates in GROMACS format"),
    ("topol.top",           "Full system topology (protein force field)"),
    ("posre.itp",           "Position restraint file for equilibration"),
]
for fname, desc in expected:
    print(f"    {fname:<25}  {desc}")

print(f"\n{'='*60}")
print(f"  Next: upload receptor PDB to Drive, then run Colab Cell 2")
print(f"{'='*60}")


CELL 6 — STAGE RECEPTOR FOR COLAB pdb2gmx

  Receptor PDB staged for Drive upload:
  Source : C:\my_projects_all\portfolio_projects\Msc_project\data\md\receptor\1J3I_chainA_GMX.pdb
  Staged : C:\my_projects_all\portfolio_projects\Msc_project\data\md\colab_upload\receptor\1J3I_chainA_GMX.pdb
  Size   : 147.9 KB

  ── Manual upload instructions ──────────────────────────
  Upload this file to Google Drive at:
    My Drive/PfDHFR_MD/receptor/1J3I_chainA_GMX.pdb

  The Colab pdb2gmx notebook will read from this path.
  ────────────────────────────────────────────────────────

  Expected outputs from Colab pdb2gmx run:
  (to be downloaded into data/md/receptor/ locally)

    1J3I_processed.gro         Protein coordinates in GROMACS format
    topol.top                  Full system topology (protein force field)
    posre.itp                  Position restraint file for equilibration

  Next: upload receptor PDB to Drive, then run Colab Cell 2


In [9]:
# ── CELL 7: Validate Receptor Topology and Document Key Parameters ──────────────

print("\n" + "=" * 60)
print("CELL 7 — VALIDATE RECEPTOR TOPOLOGY")
print("=" * 60)

def parse_gro_protein(gro_path):
    """Extract atom count, residue range and box vectors from GRO file."""
    lines   = gro_path.read_text(encoding='utf-8').splitlines()
    n_atoms = int(lines[1].strip())
    # Box vectors are on the last line
    box     = lines[-1].strip()
    # Extract residue numbers from atom lines
    res_nums = set()
    for l in lines[2:-1]:
        try:
            res_nums.add(int(l[:5]))
        except ValueError:
            pass
    return n_atoms, sorted(res_nums), box


def parse_topology(top_path):
    """
    Extract key counts from GROMACS topology:
    - Force field name
    - Number of protein atoms
    - Net charge
    - Molecule types
    """
    lines = top_path.read_text(encoding='utf-8').splitlines()

    ff_line      = ""
    n_atoms      = 0
    total_charge = 0.0
    in_atoms     = False
    molecules    = []
    in_molecules = False

    for line in lines:
        stripped = line.strip()

        # Force field
        if '#include' in line and 'forcefield' in line:
            ff_line = stripped

        # Atoms section
        if stripped.startswith('[ atoms ]'):
            in_atoms = True
            continue
        if in_atoms and stripped.startswith('['):
            in_atoms = False
        if in_atoms and stripped and not stripped.startswith(';'):
            parts = stripped.split()
            if len(parts) >= 7:
                try:
                    total_charge += float(parts[6])
                    n_atoms += 1
                except ValueError:
                    pass

        # Molecules section
        if stripped.startswith('[ molecules ]'):
            in_molecules = True
            continue
        if in_molecules and stripped.startswith('['):
            in_molecules = False
        if in_molecules and stripped and not stripped.startswith(';'):
            molecules.append(stripped)

    return {
        'forcefield'   : ff_line,
        'n_atoms'      : n_atoms,
        'total_charge' : round(total_charge, 3),
        'molecules'    : molecules,
    }


# ── Parse files ────────────────────────────────────────────────────────────────

gro_path = RECEPTOR_DIR_MD / "1J3I_processed.gro"
top_path = RECEPTOR_DIR_MD / "topol.top"

n_atoms_gro, res_nums, box_vectors = parse_gro_protein(gro_path)
top_info = parse_topology(top_path)

print(f"\n  ── GRO file ────────────────────────────────────────────")
print(f"  Total atoms     : {n_atoms_gro}")
print(f"  Residues        : {len(res_nums)}  "
      f"(range {min(res_nums)}–{max(res_nums)})")
print(f"  Box vectors     : {box_vectors}")

print(f"\n  ── Topology ────────────────────────────────────────────")
print(f"  Force field     : amber99sb-ildn")
print(f"  Water model     : TIP3P")
print(f"  Protein atoms   : {top_info['n_atoms']}")
print(f"  Net charge      : {top_info['total_charge']:+.3f} e")
print(f"  Cl⁻ ions needed : {abs(round(top_info['total_charge']))} "
      f"(to neutralise system)")
print(f"  Molecules block :")
for m in top_info['molecules']:
    print(f"    {m}")

print(f"\n  ── Resistance mutation residues (1J3I K1 strain) ───────")
resistance_residues = {51: "N51I", 59: "C59R", 108: "S108N", 164: "I164L"}
for resnum, mutation in resistance_residues.items():
    present = resnum in res_nums
    print(f"    Residue {resnum:<4}  {mutation}  "
          f"{'✓ present in structure' if present else '✗ not in range'}")

# ── Save receptor metadata ─────────────────────────────────────────────────────

receptor_meta = {
    "pdb_source"      : "1J3I",
    "chain"           : "A",
    "n_residues"      : len(res_nums),
    "residue_range"   : f"{min(res_nums)}-{max(res_nums)}",
    "n_atoms_gro"     : n_atoms_gro,
    "net_charge"      : top_info["total_charge"],
    "cl_ions_needed"  : abs(round(top_info["total_charge"])),
    "forcefield"      : "amber99sb-ildn",
    "water_model"     : "tip3p",
}
pd.DataFrame([receptor_meta]).to_csv(
    MD_DIR / "receptor_metadata.csv", index=False
)
print(f"\n  Metadata saved : {MD_DIR / 'receptor_metadata.csv'}")

print(f"\n{'='*60}")
print(f"  Next cell: write GROMACS MDP run parameter files")
print(f"{'='*60}")


CELL 7 — VALIDATE RECEPTOR TOPOLOGY

  ── GRO file ────────────────────────────────────────────
  Total atoms     : 3730
  Residues        : 223  (range 1–233)
  Box vectors     : 4.52550   6.38738   4.70296

  ── Topology ────────────────────────────────────────────
  Force field     : amber99sb-ildn
  Water model     : TIP3P
  Protein atoms   : 3730
  Net charge      : +9.000 e
  Cl⁻ ions needed : 9 (to neutralise system)
  Molecules block :
    Protein_chain_A     1

  ── Resistance mutation residues (1J3I K1 strain) ───────
    Residue 51    N51I  ✓ present in structure
    Residue 59    C59R  ✓ present in structure
    Residue 108   S108N  ✓ present in structure
    Residue 164   I164L  ✓ present in structure

  Metadata saved : C:\my_projects_all\portfolio_projects\Msc_project\data\md\receptor_metadata.csv

  Next cell: write GROMACS MDP run parameter files


In [10]:
# ── CELL 8: Write GROMACS MDP Run Parameter Files ──────────────────────────────
# Four MDP files are required for the simulation protocol:
#   em.mdp    — energy minimisation (steepest descent)
#   nvt.mdp   — NVT equilibration, 100 ps, 310 K (heat the system)
#   npt.mdp   — NPT equilibration, 100 ps, 310 K + 1 bar (pressurise)
#   md.mdp    — production MD, 50 ns, 310 K, 1 bar
#
# Parameters follow GROMACS best practices for protein-ligand systems
# with AMBER99SB-ILDN + GAFF2 + TIP3P.

print("\n" + "=" * 60)
print("CELL 8 — WRITE GROMACS MDP FILES")
print("=" * 60)

MDP_DIR.mkdir(exist_ok=True)

# ── 1. Energy Minimisation ─────────────────────────────────────────────────────
em_mdp = """\
; em.mdp — Energy Minimisation
; Steepest descent minimisation to relax clashes before MD

integrator  = steep         ; steepest descent energy minimisation
emtol       = 1000.0        ; stop when max force < 1000 kJ/mol/nm
emstep      = 0.01          ; initial step size (nm)
nsteps      = 50000         ; maximum minimisation steps

; Neighbour searching
nstlist         = 1
cutoff-scheme   = Verlet
ns_type         = grid
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
"""

# ── 2. NVT Equilibration (constant volume, heat system to 310 K) ───────────────
nvt_mdp = """\
; nvt.mdp — NVT Equilibration (100 ps)
; Constant Number, Volume, Temperature
; Purpose: heat system to 310 K with position restraints on protein + ligand

define          = -DPOSRES  ; apply position restraints (posre.itp)
integrator      = md
nsteps          = 50000     ; 50,000 steps × 2 fs = 100 ps
dt              = 0.002     ; 2 fs timestep

; Output
nstxout         = 500       ; save coordinates every 1 ps
nstvout         = 500       ; save velocities every 1 ps
nstenergy       = 500       ; save energies every 1 ps
nstlog          = 500       ; update log every 1 ps

; Neighbour searching
cutoff-scheme   = Verlet
nstlist         = 10
ns_type         = grid
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

; Electrostatics
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

; Temperature coupling (V-rescale — correct canonical ensemble)
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310

; Pressure coupling — OFF during NVT
pcoupl          = no

; Constraints
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4

; Initial velocities
gen_vel         = yes
gen_temp        = 310
gen_seed        = 42
"""

# ── 3. NPT Equilibration (constant pressure, compress box) ────────────────────
npt_mdp = """\
; npt.mdp — NPT Equilibration (100 ps)
; Constant Number, Pressure, Temperature
; Purpose: equilibrate box density at 1 bar with position restraints

define          = -DPOSRES
integrator      = md
nsteps          = 50000     ; 100 ps
dt              = 0.002

; Output
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500

; Neighbour searching
cutoff-scheme   = Verlet
nstlist         = 10
ns_type         = grid
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

; Electrostatics
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

; Temperature coupling
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310

; Pressure coupling (Parrinello-Rahman — correct NPT ensemble)
pcoupl              = Parrinello-Rahman
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com

; Constraints
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4

; Continue from NVT — no new velocities
gen_vel         = no
"""

# ── 4. Production MD (50 ns) ───────────────────────────────────────────────────
md_mdp = """\
; md.mdp — Production MD (50 ns)
; No position restraints — full system free to move
; 310 K, 1 bar, AMBER99SB-ILDN + GAFF2 + TIP3P

integrator      = md
nsteps          = 25000000  ; 25,000,000 steps × 2 fs = 50 ns
dt              = 0.002

; Output (every 10 ps)
nstxout-compressed  = 5000  ; compressed trajectory (.xtc), every 10 ps
nstxout             = 0     ; no full precision output (saves disk space)
nstvout             = 0
nstenergy           = 5000  ; energy every 10 ps
nstlog              = 5000  ; log every 10 ps

; Neighbour searching
cutoff-scheme   = Verlet
nstlist         = 10
ns_type         = grid
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

; Electrostatics
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

; Temperature coupling
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310

; Pressure coupling
pcoupl              = Parrinello-Rahman
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5

; Constraints
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4

; No velocity generation — continue from NPT
gen_vel         = no

; Dispersion correction
DispCorr        = EnerPres
"""

# ── Write files ────────────────────────────────────────────────────────────────

mdp_files = {
    "em.mdp"  : em_mdp,
    "nvt.mdp" : nvt_mdp,
    "npt.mdp" : npt_mdp,
    "md.mdp"  : md_mdp,
}

print()
for fname, content in mdp_files.items():
    path = MDP_DIR / fname
    path.write_text(content, encoding='utf-8')
    n_lines = len(content.splitlines())
    size_kb = path.stat().st_size / 1024
    print(f"  ✓  {fname:<12}  {n_lines:>3} lines  ({size_kb:.1f} KB)  → {path}")

# ── Document key simulation parameters ────────────────────────────────────────

print(f"""
  ── Simulation protocol summary ─────────────────────────
  Stage 1  Energy minimisation : steepest descent, max 50,000 steps
  Stage 2  NVT equilibration   : 100 ps, 310 K, position restrained
  Stage 3  NPT equilibration   : 100 ps, 310 K, 1 bar, position restrained
  Stage 4  Production MD       : 50 ns, 310 K, 1 bar, unrestrained
  Timestep                     : 2 fs
  Trajectory output            : every 10 ps (5,000 frames per run)
  Temperature                  : 310 K (physiological)
  Pressure                     : 1 bar
  Electrostatics               : PME (particle mesh Ewald)
  Constraints                  : h-bonds (LINCS)
  ────────────────────────────────────────────────────────
""")

print(f"  MDP files saved to: {MDP_DIR}")

print(f"\n{'='*60}")
print(f"  Next cell: assemble complete GROMACS system per compound")
print(f"{'='*60}")


CELL 8 — WRITE GROMACS MDP FILES

  ✓  em.mdp         16 lines  (0.5 KB)  → C:\my_projects_all\portfolio_projects\Msc_project\data\md\mdp_files\em.mdp
  ✓  nvt.mdp        47 lines  (1.3 KB)  → C:\my_projects_all\portfolio_projects\Msc_project\data\md\mdp_files\nvt.mdp
  ✓  npt.mdp        50 lines  (1.2 KB)  → C:\my_projects_all\portfolio_projects\Msc_project\data\md\mdp_files\npt.mdp
  ✓  md.mdp         52 lines  (1.4 KB)  → C:\my_projects_all\portfolio_projects\Msc_project\data\md\mdp_files\md.mdp

  ── Simulation protocol summary ─────────────────────────
  Stage 1  Energy minimisation : steepest descent, max 50,000 steps
  Stage 2  NVT equilibration   : 100 ps, 310 K, position restrained
  Stage 3  NPT equilibration   : 100 ps, 310 K, 1 bar, position restrained
  Stage 4  Production MD       : 50 ns, 310 K, 1 bar, unrestrained
  Timestep                     : 2 fs
  Trajectory output            : every 10 ps (5,000 frames per run)
  Temperature                  : 310 K (physiologic

In [11]:
# ── CELL 9: Assemble Protein-Ligand Complex Files ──────────────────────────────
# For each compound, combine:
#   protein GRO + ligand GRO → complex.gro  (starting coordinates)
#   topol.top + ligand ITP   → topol_complex.top  (merged topology)
#
# The complex GRO is built by appending ligand atom lines to protein atom lines
# and updating the total atom count in the header.
#
# The complex topology is built by:
#   1. Including the ligand ITP in topol.top
#   2. Adding the ligand molecule entry to the [ molecules ] section
#
# GROMACS then runs solvation, ion addition, and simulation on Colab.

print("\n" + "=" * 60)
print("CELL 9 — ASSEMBLE PROTEIN-LIGAND COMPLEX")
print("=" * 60)

def read_gro(gro_path):
    """Parse GRO file into header, atom lines, and box line."""
    lines    = gro_path.read_text(encoding='utf-8').splitlines()
    title    = lines[0]
    n_atoms  = int(lines[1].strip())
    atom_lines = lines[2:2 + n_atoms]
    box_line   = lines[2 + n_atoms]
    return title, n_atoms, atom_lines, box_line


def write_complex_gro(protein_gro, ligand_gro, out_path, ligand_name):
    """
    Merge protein and ligand GRO files into a single complex GRO.
    Ligand residue name is set to LIG for GROMACS compatibility.
    """
    _, n_prot, prot_atoms, box_line = read_gro(protein_gro)
    _, n_lig,  lig_atoms,  _        = read_gro(ligand_gro)

    # Renumber ligand atoms continuously from protein end
    # GRO format: cols 0-4 resnum, 5-9 resname, 10-14 atomname, 15-19 atomnum
    # Residue number for ligand = last protein residue + 1
    last_res = int(prot_atoms[-1][:5])
    lig_res  = (last_res + 1) % 100000   # GRO residue numbers wrap at 99999

    new_lig_atoms = []
    for i, line in enumerate(lig_atoms):
        # Replace residue number and residue name; keep atom name and coords
        atom_name = line[10:15]
        coords    = line[20:]           # x, y, z (and velocities if present)
        atom_num  = (n_prot + i + 1) % 100000
        new_line  = (f"{lig_res:>5}{'LIG':<5}{atom_name}"
                     f"{atom_num:>5}{coords}")
        new_lig_atoms.append(new_line)

    total_atoms = n_prot + n_lig
    out_lines   = (
        [f"Protein-Ligand complex: {ligand_name}"]
        + [f"{total_atoms:>5}"]
        + prot_atoms
        + new_lig_atoms
        + [box_line]
    )
    out_path.write_text("\n".join(out_lines) + "\n", encoding='utf-8')
    return n_prot, n_lig, total_atoms


def write_complex_topology(base_top, ligand_itp, out_top, ligand_name):
    """
    Merge protein topology with ligand ITP.
    Inserts #include for ligand ITP after forcefield include,
    and adds LIG molecule entry to [ molecules ] section.
    """
    top_text = base_top.read_text(encoding='utf-8')
    itp_text = ligand_itp.read_text(encoding='utf-8')

    # Extract just the [ atomtypes ] section from ITP to add to topology
    # (GROMACS requires atomtypes to be defined before moleculetype)
    itp_lines      = itp_text.splitlines()
    atomtype_lines = []
    in_atomtypes   = False
    for line in itp_lines:
        stripped = line.strip()
        if stripped == '[ atomtypes ]':
            in_atomtypes = True
            atomtype_lines.append(line)
            continue
        if in_atomtypes:
            if stripped.startswith('[') and stripped != '[ atomtypes ]':
                break
            atomtype_lines.append(line)

    atomtype_block = "\n".join(atomtype_lines) + "\n"

    # Insert atomtype block and ligand ITP include after the forcefield include
    itp_rel_path  = f"../ligand_params/{ligand_name}/{ligand_name}_GMX.itp"
    insert_block  = (f"\n{atomtype_block}\n"
                     f'; Ligand topology\n'
                     f'#include "{itp_rel_path}"\n')

    # Find insertion point — after first #include (forcefield)
    lines    = top_text.splitlines()
    inserted = False
    new_lines = []
    for line in lines:
        new_lines.append(line)
        if not inserted and line.strip().startswith('#include') and 'forcefield' in line:
            new_lines.append(insert_block)
            inserted = True

    # Add LIG to [ molecules ] section
    final_lines = []
    in_molecules = False
    for line in new_lines:
        final_lines.append(line)
        if '[ molecules ]' in line:
            in_molecules = True
        if in_molecules and line.strip().startswith('Protein_chain_A'):
            final_lines.append(f'LIG                  1')
            in_molecules = False

    out_top.write_text("\n".join(final_lines) + "\n", encoding='utf-8')


# ── Assemble complex for each compound ────────────────────────────────────────

protein_gro = RECEPTOR_DIR_MD / "1J3I_processed.gro"
base_top    = RECEPTOR_DIR_MD / "topol.top"

print()
assembly_records = []
all_ok = True

for cpd in MD_COMPOUNDS:
    name     = cpd["name"]
    itp_path = cpd.get("itp_path")
    gro_path = cpd.get("gro_path")

    # Reload paths from disk if kernel was restarted
    if itp_path is None:
        itp_path = LIGAND_DIR / name / f"{name}_GMX.itp"
        gro_path = LIGAND_DIR / name / f"{name}_GMX.gro"
        cpd["itp_path"] = itp_path
        cpd["gro_path"] = gro_path

    sys_dir = SYSTEM_DIR / name
    sys_dir.mkdir(parents=True, exist_ok=True)

    complex_gro = sys_dir / "complex.gro"
    complex_top = sys_dir / "topol_complex.top"

    print(f"  ── {name} ──────────────────────────────────────────────")

    # Build complex GRO
    try:
        n_prot, n_lig, n_total = write_complex_gro(
            protein_gro, gro_path, complex_gro, name
        )
        print(f"  GRO: protein={n_prot} + ligand={n_lig} = {n_total} atoms  ✓")
    except Exception as e:
        print(f"  GRO: FAILED — {e}")
        all_ok = False
        continue

    # Build complex topology
    try:
        write_complex_topology(base_top, itp_path, complex_top, name)
        top_size = complex_top.stat().st_size / 1024
        print(f"  TOP: {complex_top.name}  ({top_size:.1f} KB)  ✓")
    except Exception as e:
        print(f"  TOP: FAILED — {e}")
        all_ok = False
        continue

    # Copy MDP files and posre.itp into system directory for Colab convenience
    for mdp in MDP_DIR.glob("*.mdp"):
        shutil.copy(mdp, sys_dir / mdp.name)
    shutil.copy(RECEPTOR_DIR_MD / "posre.itp", sys_dir / "posre.itp")

    print(f"  MDP files and posre.itp copied to system dir  ✓")
    print(f"  System dir: {sys_dir}")
    print()

    assembly_records.append({
        "name"         : name,
        "tier"         : cpd["tier"],
        "n_prot_atoms" : n_prot,
        "n_lig_atoms"  : n_lig,
        "n_total_atoms": n_total,
        "complex_gro"  : str(complex_gro),
        "complex_top"  : str(complex_top),
    })

# ── Summary ────────────────────────────────────────────────────────────────────

n_ok = len(assembly_records)
pd.DataFrame(assembly_records).to_csv(
    MD_DIR / "assembly_summary.csv", index=False
)

print(f"  Assembly summary : {n_ok}/{len(MD_COMPOUNDS)} compounds")
print(f"  Saved            : {MD_DIR / 'assembly_summary.csv'}")
print(f"\n  {'All systems assembled.' if all_ok else 'Some assemblies failed — review above.'}")

print(f"\n{'='*60}")
print(f"  Next cell: package systems for Colab upload")
print(f"{'='*60}")


CELL 9 — ASSEMBLE PROTEIN-LIGAND COMPLEX

  ── CNP0286261_0 ──────────────────────────────────────────────
  GRO: protein=3730 + ligand=58 = 3788 atoms  ✓
  TOP: topol_complex.top  (1067.0 KB)  ✓
  MDP files and posre.itp copied to system dir  ✓
  System dir: C:\my_projects_all\portfolio_projects\Msc_project\data\md\systems\CNP0286261_0

  ── CNP0275186_1 ──────────────────────────────────────────────
  GRO: protein=3730 + ligand=63 = 3793 atoms  ✓
  TOP: topol_complex.top  (1067.4 KB)  ✓
  MDP files and posre.itp copied to system dir  ✓
  System dir: C:\my_projects_all\portfolio_projects\Msc_project\data\md\systems\CNP0275186_1

  ── CNP0539885_2 ──────────────────────────────────────────────
  GRO: protein=3730 + ligand=52 = 3782 atoms  ✓
  TOP: topol_complex.top  (1067.6 KB)  ✓
  MDP files and posre.itp copied to system dir  ✓
  System dir: C:\my_projects_all\portfolio_projects\Msc_project\data\md\systems\CNP0539885_2

  ── pyrimethamine ────────────────────────────────────────────

In [12]:
# ── CELL 10: Package Systems and Upload to Google Drive ────────────────────────
# Each compound gets a self-contained zip containing everything GROMACS needs:
#   complex.gro         — starting coordinates (protein + ligand)
#   topol_complex.top   — merged topology
#   posre.itp           — position restraints
#   em.mdp              — energy minimisation parameters
#   nvt.mdp             — NVT equilibration parameters
#   npt.mdp             — NPT equilibration parameters
#   md.mdp              — production MD parameters
#   {name}_GMX.itp      — ligand force field (referenced by topology)
#   {name}_GMX.gro      — ligand coordinates (reference)
#
# The zip is uploaded to Google Drive at:
#   My Drive/PfDHFR_MD/systems/{name}/

import zipfile
import shutil

print("\n" + "=" * 60)
print("CELL 10 — PACKAGE SYSTEMS FOR COLAB")
print("=" * 60)

# Google Drive sync folder — update if your Drive for Desktop path differs
# We write zips to colab_upload/ for manual upload if Drive for Desktop
# is not mounted
UPLOAD_DIR = MD_DIR / "colab_upload" / "systems"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print()
zip_records = []

for cpd in MD_COMPOUNDS:
    name    = cpd["name"]
    sys_dir = SYSTEM_DIR / name

    zip_path = UPLOAD_DIR / f"{name}_system.zip"

    # Files to include in zip
    files_to_zip = [
        sys_dir / "complex.gro",
        sys_dir / "topol_complex.top",
        sys_dir / "posre.itp",
        sys_dir / "em.mdp",
        sys_dir / "nvt.mdp",
        sys_dir / "npt.mdp",
        sys_dir / "md.mdp",
        LIGAND_DIR / name / f"{name}_GMX.itp",
        LIGAND_DIR / name / f"{name}_GMX.gro",
    ]

    # Verify all files exist before zipping
    missing = [f for f in files_to_zip if not f.exists()]
    if missing:
        print(f"  ✗  {name} — missing files:")
        for m in missing:
            print(f"       {m.name}")
        continue

    # Create zip
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files_to_zip:
            zf.write(f, arcname=f.name)   # flat structure inside zip

    zip_kb = zip_path.stat().st_size / 1024
    print(f"  ✓  {name:<22}  {zip_kb:.1f} KB  → {zip_path.name}")

    zip_records.append({
        "name"     : name,
        "zip_path" : str(zip_path),
        "zip_kb"   : round(zip_kb, 1),
        "n_files"  : len(files_to_zip),
    })

# ── Summary ────────────────────────────────────────────────────────────────────

print(f"""
  ── Upload instructions ──────────────────────────────────
  Upload each zip to Google Drive at:
    My Drive/PfDHFR_MD/systems/

  So the Drive structure should be:
    PfDHFR_MD/
      systems/
        CNP0286261_0_system.zip
        CNP0275186_1_system.zip
        CNP0539885_2_system.zip
        pyrimethamine_system.zip
  ────────────────────────────────────────────────────────
""")

pd.DataFrame(zip_records).to_csv(
    MD_DIR / "zip_manifest.csv", index=False
)
print(f"  Manifest saved : {MD_DIR / 'zip_manifest.csv'}")
print(f"  Zips located   : {UPLOAD_DIR}")

print(f"\n{'='*60}")
print(f"  Next: upload zips to Drive, then run Colab MD notebook")
print(f"{'='*60}")


CELL 10 — PACKAGE SYSTEMS FOR COLAB

  ✓  CNP0286261_0            208.4 KB  → CNP0286261_0_system.zip
  ✓  CNP0275186_1            209.5 KB  → CNP0275186_1_system.zip
  ✓  CNP0539885_2            208.4 KB  → CNP0539885_2_system.zip
  ✓  pyrimethamine           204.4 KB  → pyrimethamine_system.zip

  ── Upload instructions ──────────────────────────────────
  Upload each zip to Google Drive at:
    My Drive/PfDHFR_MD/systems/

  So the Drive structure should be:
    PfDHFR_MD/
      systems/
        CNP0286261_0_system.zip
        CNP0275186_1_system.zip
        CNP0539885_2_system.zip
        pyrimethamine_system.zip
  ────────────────────────────────────────────────────────

  Manifest saved : C:\my_projects_all\portfolio_projects\Msc_project\data\md\zip_manifest.csv
  Zips located   : C:\my_projects_all\portfolio_projects\Msc_project\data\md\colab_upload\systems

  Next: upload zips to Drive, then run Colab MD notebook


In [13]:
# ── CELL 11: Draft Methods Text and Save Notebook 05a Handover ─────────────────

print("\n" + "=" * 60)
print("CELL 11 — METHODS TEXT AND HANDOVER RECORD")
print("=" * 60)

# ── Methods text — Paper 3, Section 2.10 ──────────────────────────────────────

methods_text = """
PAPER 3 METHODS — Section 2.10: Molecular Dynamics Simulations
===============================================================

2.10.1 Compound Selection for MD
Three compounds were selected from the molecular docking results for
molecular dynamics simulation: CNP0286261.0 (African NP, -11.86 kcal/mol),
CNP0275186.1 (African NP, -11.13 kcal/mol), and CNP0539885.2 (Global NP,
-10.03 kcal/mol, p_active=0.615). Pyrimethamine (-6.90 kcal/mol) was
included as a reference control, as it is the co-crystallised ligand
analogue in PDB structure 1J3I. Selection criteria were: docking score
<= -9.0 kcal/mol, Tanimoto diversity (Tc < 0.40, ECFP4), and
representation of both African NP and global NP chemical space.

2.10.2 System Preparation
Docked binding poses (mode 1, lowest energy) were extracted from AutoDock
Vina output files using a custom Python parser. Heavy-atom coordinates
were assigned from the docked pose; hydrogen atoms were placed using RDKit
ETKDGv3 embedding followed by MMFF94 minimisation with all heavy atoms
fixed, ensuring docked geometry was preserved.

Ligand force field parameters were generated using ACPYPE v.2023.10.27
with the GAFF2 atom type scheme and AM1-BCC partial charges computed via
the sqm programme from AmberTools. All four ligands were assigned a net
formal charge of zero. Parametrisation was performed on a Linux environment
(Google Colab) due to platform constraints of AmberTools.

The protein receptor (PfDHFR-TS K1 strain, PDB: 1J3I, chain A) was
prepared using GROMACS pdb2gmx (version 2025.4) with the AMBER99SB-ILDN
force field and TIP3P water model. Hydrogen atoms were added by pdb2gmx;
the protein net charge was determined to be +9 e, requiring addition of
9 Cl- counter-ions for system neutralisation. Protein-ligand complex
coordinate files were assembled by merging protein and ligand GRO files,
with ligand residues labelled LIG. Complex topology files were generated
by incorporating GAFF2 ligand parameters into the AMBER99SB-ILDN protein
topology.

2.10.3 Simulation Protocol
All MD simulations were performed using GROMACS 2025.4 on GPU-accelerated
infrastructure (Google Colab, NVIDIA T4 GPU). Each protein-ligand complex
was solvated in a dodecahedral TIP3P water box with a minimum solute-to-box
edge distance of 1.0 nm. Na+ and Cl- ions were added to neutralise the
system and achieve a physiological salt concentration of 0.15 M. Long-range
electrostatics were treated with the Particle Mesh Ewald (PME) method
(fourth-order interpolation, 0.16 nm Fourier spacing). Van der Waals and
electrostatic real-space cutoffs were set to 1.0 nm. Covalent bonds
involving hydrogen atoms were constrained using the LINCS algorithm
(order 4).

Each system underwent a four-stage simulation protocol:
(i)   Steepest descent energy minimisation (maximum 50,000 steps, force
      convergence < 1000 kJ mol-1 nm-1) to remove steric clashes;
(ii)  NVT equilibration (100 ps, 310 K) with position restraints applied
      to all protein and ligand heavy atoms (force constant 1000 kJ mol-1
      nm-2), using the V-rescale thermostat (time constant 0.1 ps);
(iii) NPT equilibration (100 ps, 310 K, 1 bar) with position restraints,
      using the Parrinello-Rahman barostat (time constant 2.0 ps,
      compressibility 4.5e-5 bar-1);
(iv)  Production MD (50 ns, 310 K, 1 bar, unrestrained) with a 2 fs
      timestep. Trajectory frames were saved every 10 ps (5,000 frames
      per simulation).

2.10.4 Analysis
Trajectory analysis was performed using GROMACS built-in tools and MDAnalysis.
The following metrics were computed for each simulation:
- Backbone RMSD relative to the energy-minimised structure
- Per-residue RMSF of binding site residues (within 5 A of ligand)
- Radius of gyration of the protein
- Protein-ligand hydrogen bond occupancy (distance <= 3.5 A, angle >= 120 deg)
- MM-PBSA binding free energy estimation using gmx_MMPBSA
All metrics were compared across the three hit compounds and pyrimethamine
reference to assess relative binding stability.
"""

methods_path = BASE / "paper3_methods_section2_10.txt"
methods_path.write_text(methods_text, encoding='utf-8')
print(f"\n  Methods text saved : {methods_path}")

# ── File inventory ─────────────────────────────────────────────────────────────

print(f"\n  ── Notebook 05a file inventory ─────────────────────────")

inventory = {
    "Pose PDB files (4)"     : POSES_DIR,
    "ACPYPE ITP files (4)"   : LIGAND_DIR,
    "ACPYPE GRO files (4)"   : LIGAND_DIR,
    "Receptor PDB"           : RECEPTOR_DIR_MD / "1J3I_chainA_GMX.pdb",
    "Receptor GRO"           : RECEPTOR_DIR_MD / "1J3I_processed.gro",
    "Receptor topology"      : RECEPTOR_DIR_MD / "topol.top",
    "Position restraints"    : RECEPTOR_DIR_MD / "posre.itp",
    "MDP files (4)"          : MDP_DIR,
    "Complex GRO files (4)"  : SYSTEM_DIR,
    "Complex TOP files (4)"  : SYSTEM_DIR,
    "System zips (4)"        : UPLOAD_DIR,
}

for label, path in inventory.items():
    exists = path.exists() if isinstance(path, Path) else True
    print(f"  {'✓' if exists else '✗'}  {label}")

print(f"""
  ── Notebook 05b instructions (Colab MD execution) ───────
  1. Open a new Colab notebook named: md_execution
  2. Mount Google Drive (same account)
  3. Install GROMACS:
       conda install -c conda-forge gromacs -y
  4. For each compound zip in Drive/PfDHFR_MD/systems/:
       a. Unzip into /content/md_work/{name}/
       b. Run gmx solvate → gmx genion → gmx grompp → gmx mdrun
          for each of the 4 stages (em, nvt, npt, md)
       c. Save trajectory and energy files to Drive after each stage
  5. Download trajectory files for local analysis in Notebook 05c
  ────────────────────────────────────────────────────────
""")

print(f"{'='*60}")
print(f"  Notebook 05a COMPLETE")
print(f"  Next: notebook_05b_md_colab_execution.ipynb")
print(f"{'='*60}")


CELL 11 — METHODS TEXT AND HANDOVER RECORD

  Methods text saved : C:\my_projects_all\portfolio_projects\Msc_project\paper3_methods_section2_10.txt

  ── Notebook 05a file inventory ─────────────────────────
  ✓  Pose PDB files (4)
  ✓  ACPYPE ITP files (4)
  ✓  ACPYPE GRO files (4)
  ✓  Receptor PDB
  ✓  Receptor GRO
  ✓  Receptor topology
  ✓  Position restraints
  ✓  MDP files (4)
  ✓  Complex GRO files (4)
  ✓  Complex TOP files (4)
  ✓  System zips (4)

  ── Notebook 05b instructions (Colab MD execution) ───────
  1. Open a new Colab notebook named: md_execution
  2. Mount Google Drive (same account)
  3. Install GROMACS:
       conda install -c conda-forge gromacs -y
  4. For each compound zip in Drive/PfDHFR_MD/systems/:
       a. Unzip into /content/md_work/pyrimethamine/
       b. Run gmx solvate → gmx genion → gmx grompp → gmx mdrun
          for each of the 4 stages (em, nvt, npt, md)
       c. Save trajectory and energy files to Drive after each stage
  5. Download traject